In [1]:
from transformers import pipeline
import  json
import pandas as pd
from tqdm import tqdm
from ch_test import prepare_unique_sentences, LANG_MAP, LANG_MAP_INV


KeyboardInterrupt: 

In [2]:
phoneme_model = "facebook/wav2vec2-xlsr-53-espeak-cv-ft"
eval_batch_size = 16
transcriber = pipeline(model=phoneme_model, device=0, batch_size=eval_batch_size)

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

D:\miniconda3\envs\coqui_tts\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\cluster\data\deri\hf_cache\hub\models--facebook--wav2vec2-xlsr-53-espeak-cv-ft. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Device set to use cuda:0


In [3]:
import os
conditioning_bpath = "/02Datasets/02 Audio Processing/snf_test/wav"
conditioning_sid = 'ce50fa32-dbc4-4f72-bd5f-5128784a5abc'
conditioning_audios = [
    '1e65bc1316280785b1083ec0f4a3383712fe7f94159671f1bcfd6341e7ce605d',
    '359287ecc8dd219c23f8a02e8822cee145da46acc2872687c18838589ef56ff6',
    '35cf58e4ac663e3cfaf694fd6113828e5fe006b54aa1b5bf13810f6c8f51cfeb',
    '36735e15f0f119ac84c4ab73d8d95a6a3ab6c3f53c8d783bf321a38d10274df2',
    '3b71c15a510ed6963a39151f36b1818311a1cf19232d54588f8856ceb0d1133c'
]

conditioning_paths = [os.path.join(conditioning_bpath, conditioning_sid, f"{audio}.wav") for audio in conditioning_audios]

In [8]:
out_strs = transcriber(conditioning_paths)
out_strs = [out_str['text'] for out_str in out_strs]

In [8]:
import joblib
model_path = 'dial_model/text_clf_3.joblib'
class_nr_map = {"Bern": 6, "Wallis": 2, "Basel": 5, "Graubünden": 4, "Ostschweiz": 3, "Zürich": 0, "Innerschweiz": 1, "Deutschland": 7}
inverse_cls_map = {v: k for k, v in class_nr_map.items()}

model = joblib.load(model_path)

In [10]:
preds = model.predict(out_strs)
preds

array([6, 5, 5, 5, 6])

In [11]:
concat_str = ' '.join(out_strs)
pred = model.predict([concat_str])
pred

array([5])

In [11]:
cname = "gen_config"

with open(f'{cname}.json', 'rt', encoding='utf-8') as f:
    config = json.load(f)


In [27]:
device = "cuda"

model_name = config["model_name"]
output_bpath = os.path.join(config["output_bpath"], model_name)
dataspeech_stats_path = config["dataspeech_stats_path"]
dataspeech_stats_fname = config["dataspeech_stats_fname"]
test_sentence_path = config["test_sentence_path"]
test_sentence_fname = config["test_sentnece_fname"]
n_samples = config["n_samples"]
speaker_ref_path = config["speaker_ref_path"]

dataspeech_path = os.path.join(dataspeech_stats_path, dataspeech_stats_fname)
test_sentence_full_path = os.path.join(test_sentence_path, test_sentence_fname)

randdf = pd.read_csv(dataspeech_path, sep='\t')
unique_test_sentence_df = prepare_unique_sentences(test_sentence_full_path, k=n_samples)

unique_speakers = randdf['speaker_id'].unique()
output_data = []


for sid, speaker in tqdm(enumerate(unique_speakers)):
    for dial_tag in LANG_MAP.keys():
        opath_speaker = os.path.join(output_bpath, speaker)
        orig_dialect = randdf[randdf['speaker_id'] == speaker]['dialect'].iloc[0]
        orig_dialect_tag = LANG_MAP_INV[orig_dialect]
        all_audio_files = []
        for idx, row in unique_test_sentence_df.iterrows():
            audio_file = os.path.join(opath_speaker, dial_tag, f'sent-{idx}.wav')
            if not os.path.exists(audio_file):
                print(f"Skipping {opath_speaker}-{dial_tag}-{idx}")
                continue
            all_audio_files.append(audio_file)
        if len(all_audio_files) == 0:
            continue
        out_strs = transcriber(all_audio_files)
        out_strs = [out_str['text'] for out_str in out_strs]
        
        #concat all transcriptions
        concat_str = ' '.join(out_strs)
        long_pred =model.predict([concat_str])
        
        #concat batches of 10 transcriptions and predict
        batch_size = 10
        n_batches = len(out_strs) // batch_size
        preds = []
        for i in range(n_batches):
            start = i * batch_size
            end = (i + 1) * batch_size
            batch_cat = ' '.join(out_strs[start:end])
            pred = model.predict([batch_cat])
            preds.extend(pred)
        
        #max vote over preds
        pred = int(max(set(preds), key=preds.count))
        output_data.append({
            "speaker_id": speaker,
            "dialect": orig_dialect,
            "dialect_tag": orig_dialect_tag,
            "pred": pred,
            "pred_tag": inverse_cls_map[pred],
            "long_pred": long_pred[0],
            "long_pred_tag": inverse_cls_map[long_pred[0]]
        })
        print(f"Speaker {speaker} - {orig_dialect_tag} - {dial_tag} - {pred} - {long_pred[0]}")

0it [00:00, ?it/s]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
C:\Users\Jan Deriu\AppData\Local\Temp\ipykernel_31508\1530833264.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  long_pred = int(model.predict([concat_str]))
0it [01:21, ?it/s]


TypeError: 'int' object is not subscriptable

In [26]:
#get class map of model
model.classes_

array([0, 1, 2, 3, 4, 5, 6, 7])

In [12]:
#output_df.to_csv(os.path.join(output_bpath, 'dialect_preds.tsv'), sep='\t', index=False) load this
import os

model_name = config["model_name"]
output_bpath = os.path.join(config["output_bpath"], model_name)
output_df = pd.read_csv(os.path.join(output_bpath, 'dialect_preds.tsv'), sep='\t')
output_df

,speaker_id,dialect,dialect_tag,cond_dialect,pred,pred_tag,long_pred,long_pred_tag
0,031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72,Graubünden,ch_gr,ch_be,6,ch_be,6,ch_be
1,031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72,Graubünden,ch_gr,ch_bs,5,ch_bs,5,ch_bs
2,031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72,Graubünden,ch_gr,ch_gr,3,ch_gr,3,ch_gr
3,031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72,Graubünden,ch_gr,ch_in,1,ch_in,1,ch_in
4,031b0a74-5bdd-47e7-b8b7-9bb58d0e8c72,Graubünden,ch_gr,ch_os,4,ch_os,4,ch_os
...,...,...,...,...,...,...,...,...
211,fb1b67be-8d8f-47bb-b15c-ee1138f0d4ac,Wallis,ch_vs,ch_in,1,ch_in,1,ch_in
212,fb1b67be-8d8f-47bb-b15c-ee1138f0d4ac,Wallis,ch_vs,ch_os,4,ch_os,4,ch_os
213,fb1b67be-8d8f-47bb-b15c-ee1138f0d4ac,Wallis,ch_vs,ch_vs,2,ch_vs,2,ch_vs
214,fb1b67be-8d8f-47bb-b15c-ee1138f0d4ac,Wallis,ch_vs,ch_zh,1,ch_in,1,ch_in


In [13]:
#compute the F1 score between cond_dialect and the long_pred_tag
from sklearn.metrics import f1_score

for dial_tag in LANG_MAP.keys():
    _dtag = output_df[output_df['cond_dialect'] == dial_tag]
    _score = f1_score(_dtag['cond_dialect'], _dtag['pred_tag'], average='weighted')
    print(f"{dial_tag}\t{_score}")

ch_be	0.9615384615384616
ch_bs	1.0
ch_gr	1.0
ch_in	1.0
ch_os	1.0
ch_vs	1.0
ch_zh	0.2
de	0.0


In [14]:
#compute the confusion matrix between cond_dialect and the long_pred_tag for overall output_df

from sklearn.metrics import confusion_matrix

sorted_labels = sorted(LANG_MAP.keys())

conf_mat = confusion_matrix(output_df['cond_dialect'], output_df['pred_tag'], labels=sorted_labels)
#compute overall f1-score without the "de" tag
no_de = output_df[output_df['cond_dialect'] != "de"]
_score = f1_score(no_de['cond_dialect'], no_de['pred_tag'], average='weighted')
print(f"Overall F1 score: {_score}")


Overall F1 score: 0.8353420772775612


In [10]:
conf_mat, sorted_labels

(array([[27,  0,  0,  0,  0,  0,  0,  0],
        [ 0, 26,  0,  0,  1,  0,  0,  0],
        [ 0,  0, 27,  0,  0,  0,  0,  0],
        [ 0,  0,  0, 26,  0,  0,  1,  0],
        [ 0,  0,  0,  0, 27,  0,  0,  0],
        [ 0,  0,  0,  0,  0, 27,  0,  0],
        [ 0,  0,  0, 26,  0,  0,  1,  0],
        [ 0,  0,  0,  0,  0,  0,  0, 27]]),
 ['ch_be', 'ch_bs', 'ch_gr', 'ch_in', 'ch_os', 'ch_vs', 'ch_zh', 'de'])